# AI Project

User Detection Pipeline

In [1]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("YOLO11 loaded successfully")

YOLO11 loaded successfully


In [ ]:
# Imports
import cv2
import time
from ultralytics import YOLO

# Open webcam, typically index 0
cap = cv2.VideoCapture(0)

# Set trigger rules deciding when someone is a real user
MIN_BOX_AREA = 120000 # Minimum boundin box area to consider a person a potential user
HOLD_TIME = 3.0 # Time a potential user must wait before being considered a user
RESET_TIME = 2.0 # Time after the user leaves before system resets

candidate_time = None # Records time when a potential user was first detected
last_qualified_seen = 0 # Records last time a potential user was detected
greeted = False # Keeps track of greetings so user is only greeted once

# Main loop, runs continuously until quit
while True:
    # Read a frame from the camera, ret says if this was successful and frame is the image
    ret, frame = cap.read()
    if not ret:
        print("Camera not working") # Useful for debugging
        break

    # Run YOLO object detection, send frame to model, class 0 is for the person class
    results = model(frame, classes=[0], verbose=False)
    result = results[0] # store detection results for this frame

    # Store the largest bounding box and its area
    largest_box = None
    largest_area = 0

    # Loopp over all detected people
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist()) # Get coirdinates of bounding box
        area = (x2 - x1) * (y2 - y1) # calculate its area using the coordinates

        # Keep only the largest bounding box, after the loop you will have the biggest detected person in the frame
        if area > largest_area:
            largest_area = area
            largest_box = (x1, y1, x2, y2)

    now = time.time() # Get current time in seconds

    # If statement to check if a valid potential user is detected
    if largest_box is not None and largest_area >= MIN_BOX_AREA:
        last_qualified_seen = now # Store time last potewntial user was seen (updates every frame)

        if candidate_time is None: # Store time potewntial user first appears
            candidate_time = now

        if not greeted and (now - candidate_time >= HOLD_TIME): # If potential user has been present for lonmg enough they are now a user, if system hasent greeted them already do so (Prompt LLM to greet user)
            greeted = True
            print("User detected -> trigger greeting")
    else:
        candidate_time = None # If no valid user present simply do nothing

        # If a user was greeted and no user has been seen for long enough reset system to be ready for new user
        if greeted and (now - last_qualified_seen >= RESET_TIME):
            greeted = False
            print("System reset")

    frame = result.plot() # Draw YOLO detections on the frame

    # If there is a largest bounding box (There will always be one if at least one person is detected) draw the area of the largest box on the frame, this is useful for debugging and understanding how the system works
    if largest_box is not None:
        x1, y1, x2, y2 = largest_box # Extract coordinates of largest box
        # Draw text on fram
        cv2.putText(
            frame, # Draw text on current video frame
            f"Largest area: {largest_area}", # Print largest area and number to screen
            (x1+150, max(30, y1 - 5)), # Where text appears on screen
            cv2.FONT_HERSHEY_SIMPLEX, # Font style
            0.7, # font size
            (0, 0, 0), # Text color
            2 # Text thickness
        )

    # If the user has been greeted and a largest box is detected, draw the area of the largest box on the frame, this is useful for debugging and understanding how the system works
    if greeted == True and largest_box is not None:
        x1, y1, x2, y2 = largest_box # Extract coordinates of largest box
        # Draw text on fram
        cv2.putText(
            frame, # Draw text on current video frame
            f"User", # Print largest area and number to screen
            (x1, max(30, y1 + 20)), # Where text appears on screen
            cv2.FONT_HERSHEY_SIMPLEX, # Font style
            0.7, # font size
            (0, 0, 0), # Text color
            2 # Text thickness
        )

    

    cv2.imshow("User Detection", frame) # Open window showing feed

    if cv2.waitKey(1) == ord("q"): # Leave loop if q is pressed
        break

# Clean up
cap.release()
cv2.destroyAllWindows()

User detected -> trigger greeting
System reset
User detected -> trigger greeting
System reset
User detected -> trigger greeting
